In [13]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import re

## Major Sectors in the FTSE 100 (As of 2025)

In [6]:
url = "http://ftse100index.com/ftse-100-sector-breakdown/"
headers = {"User-Agent": "Mozilla/5.0"}

try:
    page = requests.get(url, headers=headers, timeout=10)
    page.raise_for_status()
    
    print(f"Request Successful. Status code: {page.status_code}\n")
    
    soup = BeautifulSoup(page.content, 'lxml')

    table = soup.find_all('table')
    if table is None:
        raise ValueError("No table found on the page.")
    
    majorSectors  = pd.read_html(StringIO(str(table)))[0]

    majorSectors.columns = (
        majorSectors.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    print("Columns found:", list(majorSectors.columns), "\n")
    sectors = majorSectors.copy()

    col_map = {
        "Sector": [c for c in majorSectors.columns if "Sector" in c][0],
        "Approx Weight": [c for c in majorSectors.columns if "Approx. Weight" in c][0],
        "Notable Companies": [c for c in majorSectors.columns if "Notable Companies" in c][0],
    }

    majorSectors = majorSectors[[col_map["Sector"], col_map["Approx Weight"], col_map["Notable Companies"]]]
    majorSectors.columns = ["Sector", "Approx Weight", "Notable Companies"]

    #display(majorSectors.head())
    
except requests.exceptions.Timeout:
    print("The request timed out. Please check your network or try again later.")
except requests.exceptions.HTTPError as http_err:
    print(f"HTTP error occurred: {http_err}")
except ValueError as val_err:
    print(f"Value error: {val_err}")
except Exception as e:
    print(f"Unexpected error: {e}")

Request Successful. Status code: 200

Columns found: ['Sector', 'Approx. Weight', 'Notable Companies'] 



In [7]:
majorSectors

,Sector,Approx Weight,Notable Companies
0,Energy,~15%,"Shell, BP"
1,Financials,~18%,"HSBC, Lloyds, Barclays"
2,Healthcare,~12%,"AstraZeneca, GSK"
3,Consumer Goods,~11%,"Unilever, Diageo"
4,Industrials,~10%,"Rolls-Royce, BAE Systems"
5,Utilities,~5%,"National Grid, SSE"
6,Telecoms,~4%,"Vodafone, BT Group"
7,Real Estate,~2%,"Land Securities, British Land"
8,Technology,~2%,"Sage Group, Aveva"
9,Basic Materials,~9%,"Rio Tinto, Anglo American"


In [8]:
majorSectors2 = majorSectors.copy()

majorSectors2["Notable Companies"] = (
    majorSectors2["Notable Companies"]
    .astype(str)
    .str.split(",")
)

majorSectors2 = majorSectors2.explode("Notable Companies")
majorSectors2["Notable Companies"] = majorSectors2["Notable Companies"].str.strip()
display(majorSectors2)

,Sector,Approx Weight,Notable Companies
0,Energy,~15%,Shell
0,Energy,~15%,BP
1,Financials,~18%,HSBC
1,Financials,~18%,Lloyds
1,Financials,~18%,Barclays
2,Healthcare,~12%,AstraZeneca
2,Healthcare,~12%,GSK
3,Consumer Goods,~11%,Unilever
3,Consumer Goods,~11%,Diageo
4,Industrials,~10%,Rolls-Royce


In [14]:
url = "https://en.wikipedia.org/wiki/FTSE_100_Index"
headers = {"User-Agent": "Mozilla/5.0"}

page = requests.get(url, headers=headers, timeout=10)
page.raise_for_status()

soup = BeautifulSoup(page.content, "lxml")

tables = soup.find_all("table", {"class": "wikitable"})

tickerTable = tables[3]  

tickerDf = pd.read_html(StringIO(str(tickerTable)))[0]

tickerDf.columns = (
    tickerDf.columns.astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

#print(tickerDf.columns)

company_col = [c for c in tickerDf.columns if "Company" in c][0]
ticker_col  = [c for c in tickerDf.columns if "Ticker" in c][0]
sector_col  = [c for c in tickerDf.columns if "FTSE industry" in c or "benchmark sector" in c][0]

tickerDf2 = tickerDf[[company_col, ticker_col, sector_col]].rename(
    columns={
        company_col: "Company",
        ticker_col: "Ticker",
        sector_col: "Sector"
    }
)
tickerDf3 = tickerDf2.copy()

tickerDf3["Ticker_Yahoo"] = tickerDf3["Ticker"].astype(str).str.strip() + ".L"

In [11]:
display(tickerDf2)
display(tickerDf3)

,Company,Ticker,Sector
0,3i,III,Financial services
1,Admiral Group,ADM,Insurance
2,Airtel Africa,AAF,Telecommunications services
3,Alliance Witan,ALW,Investment Trusts
4,Anglo American plc,AAL,Mining
...,...,...,...
95,United Utilities,UU,Multiline utilities
96,Vodafone Group,VOD,Mobile telecommunications
97,Weir Group,WEIR,Industrial goods and services
98,Whitbread,WTB,Retail hospitality


,Company,Ticker,Sector,Ticker_Yahoo
0,3i,III,Financial services,III.L
1,Admiral Group,ADM,Insurance,ADM.L
2,Airtel Africa,AAF,Telecommunications services,AAF.L
3,Alliance Witan,ALW,Investment Trusts,ALW.L
4,Anglo American plc,AAL,Mining,AAL.L
...,...,...,...,...
95,United Utilities,UU,Multiline utilities,UU.L
96,Vodafone Group,VOD,Mobile telecommunications,VOD.L
97,Weir Group,WEIR,Industrial goods and services,WEIR.L
98,Whitbread,WTB,Retail hospitality,WTB.L


In [17]:
majorSectorsMerge = majorSectors2.merge(
    tickerDf3,
    left_on="Notable Companies",
    right_on="Company",
    how="left"
)

majorSectorsMerged = majorSectorsMerge.copy()
unmatched = majorSectorsMerged[majorSectorsMerged["Ticker"].isna()].copy()

for i, row in unmatched.iterrows():
    notable = row["Notable Companies"]
    pattern = rf"\b{re.escape(str(notable))}\b"
    candidates = tickerDf3[tickerDf3["Company"].str.contains(pattern, case=False, na=False)]
    
    if len(candidates) == 1:
        x = candidates.iloc[0]
        majorSectorsMerged.loc[i, "Company"] = x["Company"]
        majorSectorsMerged.loc[i, "Ticker"] = x["Ticker"]
        majorSectorsMerged.loc[i, "Ticker_Yahoo"] = x["Ticker_Yahoo"] 

In [22]:
majorSectorsMerged["Current_constituent"] = ~majorSectorsMerged["Ticker"].isna()
majorSectorsMerged.columns = ['Sector', 'Approx Weight', 'Notable Companies', 'Company', 'Ticker',
       'Sector_y', 'Ticker_Yahoo', 'Current_constituent']
display(majorSectorsMerged[["Sector", 'Approx Weight', "Notable Companies", "Ticker", "Ticker_Yahoo", "Current_constituent"]])

,Sector,Approx Weight,Notable Companies,Ticker,Ticker_Yahoo,Current_constituent
0,Energy,~15%,Shell,SHEL,SHEL.L,True
1,Energy,~15%,BP,BP,BP.L,True
2,Financials,~18%,HSBC,HSBA,HSBA.L,True
3,Financials,~18%,Lloyds,LLOY,LLOY.L,True
4,Financials,~18%,Barclays,BARC,BARC.L,True
5,Healthcare,~12%,AstraZeneca,AZN,AZN.L,True
6,Healthcare,~12%,GSK,GSK,GSK.L,True
7,Consumer Goods,~11%,Unilever,ULVR,ULVR.L,True
8,Consumer Goods,~11%,Diageo,DGE,DGE.L,True
9,Industrials,~10%,Rolls-Royce,RR,RR.L,True
